In [0]:
%pip install confluent-kafka

In [0]:
from pyspark.sql.functions import col, from_json, cast
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType, TimestampType

In [0]:
confluent_api_key = dbutils.secrets.get(scope = 'silk_pipeline', key = 'confluent_api_key')
confluent_api_secret = dbutils.secrets.get(scope = 'silk_pipeline', key = 'confluent_api_secret')
bootstrap_server = dbutils.secrets.get(scope = 'silk_pipeline', key = 'confluent_bootstrap_server')

In [0]:
# Read from Kafka topic as a stream
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap_server) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{confluent_api_key}" '
        f'password="{confluent_api_secret}";'
    ) \
    .option("subscribe", "silk-prices") \
    .option("startingOffsets", "earliest") \
    .load()

In [0]:
# {
#   "timestamp": "2026-06-02T18:23:12.152679",
#   "ppi_date": "2026-04-01",
#   "ppi_value": 240.218,
#   "price_usd_mt": 140955.33,
#   "currency": "USD",
#   "unit": "per MT",
#   "source": "FRED WPU034203 + simulation"
# }

schema = StructType([
    StructField("timestamp",    TimestampType(),  True),
    StructField("ppi_date",     DateType(),  True),
    StructField("ppi_value",    DoubleType(),  True),
    StructField("price_usd_mt", DoubleType(),  True),
    StructField("currency",     StringType(),  True),
    StructField("unit",         StringType(),  True),
    StructField("source",       StringType(),  True)
])

parsed_stream = kafka_stream \
    .select(
        col("value").cast("string").alias("raw_json")
    ) \
    .select(
        from_json(col("raw_json"), schema).alias("data")
    ) \
    .select("data.*")  # Flatten nested struct into columns

In [0]:
query = parsed_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/default/titanic/Data/silk_checkpoint") \
    .trigger(availableNow=True) \
    .toTable("silk_prices")

query.awaitTermination()

spark.sql("Delete from silk_prices where price_usd_mt>140000")
print("Consumer finished, all available messages processed")


Consumer finished, all available messages processed


In [0]:
# Check the Delta Table

df = spark.table("silk_prices")
df.printSchema()
df.show(truncate=False)
print(f"Total rows: {df.count()}")



root
 |-- timestamp: timestamp (nullable = true)
 |-- ppi_date: date (nullable = true)
 |-- ppi_value: double (nullable = true)
 |-- price_usd_mt: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- source: string (nullable = true)

+--------------------------+----------+---------+------------+--------+------+---------------------------+
|timestamp                 |ppi_date  |ppi_value|price_usd_mt|currency|unit  |source                     |
+--------------------------+----------+---------+------------+--------+------+---------------------------+
|2026-06-02 18:25:32.522321|2026-04-01|240.218  |59327.6     |USD     |per MT|FRED WPU034203 + simulation|
|2026-06-03 07:55:41.418677|2026-04-01|240.218  |59809.23    |USD     |per MT|FRED WPU034203 + simulation|
|2026-06-03 07:55:52.974906|2026-04-01|240.218  |59809.23    |USD     |per MT|FRED WPU034203 + simulation|
|2026-06-03 07:55:56.117663|2026-04-01|240.218  |59809.23    |USD     |